## How to use this notebook

This notebook tells a story, step by step, using **live data**. To follow along, click into each cell and press the ▶ *play* button (or `Shift + Enter`) to run it — then move to the next one.

A few things to know before you start:
- **Run cells in order, top to bottom.** Later cells depend on data and variables set up by earlier ones, so skipping around or running things out of sequence will cause errors.
- **Some cells change state** (e.g. moving data between storage tiers, adjusting Kafka settings) rather than just reading it. Re-running the notebook from the top, rather than re-running a cell in isolation, is the safest way to see it again.
- Expect the full walkthrough to take around **10 minutes**.


## Table of contents

- [Glossary](#glossary)
- [Overview](#overview)
- [Part 1 — The Problem](#part-1-the-problem)
- [Part 2 — Enter Streambased](#part-2-enter-streambased)
- [Part 3 — Under the Hood: Composable Views](#part-3-under-the-hood-composable-views)
- [Part 4 — Hot-to-Cold Transfer](#part-4-hot-to-cold-transfer)
- [Part 5 — Kafka Consumers Reading Archived Data (Slipstream)](#part-5-kafka-consumers-reading-archived-data)
- [Part 6 — CDC: Automatic Change-Data-Capture Materialization](#part-6-cdc-materialization)
- [Summary](#summary)


<a id="overview"></a>
## Overview


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<!-- ── HEADER ───────────────────────────────────────────────────────── -->
<div style="text-align: center; margin-bottom: 24px;">
  <div style="font-size: 0.78em; letter-spacing: 3px; color: #90a4ae;
              text-transform: uppercase; margin-bottom: 8px;">Demo environment</div>
  <h2 style="font-size: 1.8em; font-weight: 800; color: #1a237e; margin: 0;">
    Architecture Overview
  </h2>
</div>

<!-- ── DIAGRAM ───────────────────────────────────────────────────────── -->
<div style="background: #f8f9fa; border: 1.5px solid #e0e0e0; border-radius: 14px;
            padding: 32px 24px; margin-bottom: 16px;">
<svg viewBox="0 0 760 300" xmlns="http://www.w3.org/2000/svg"
     style="width: 100%; max-width: 760px; display: block; margin: 0 auto;">

  <!-- ── Layer labels ── -->
  <text x="14" y="60"  font-size="9" fill="#90a4ae" font-family="system-ui"
        text-anchor="middle" transform="rotate(-90,14,60)">YOUR INFRA</text>
  <text x="14" y="175" font-size="9" fill="#90a4ae" font-family="system-ui"
        text-anchor="middle" transform="rotate(-90,14,175)">STREAMBASED</text>
  <text x="14" y="268" font-size="9" fill="#90a4ae" font-family="system-ui"
        text-anchor="middle" transform="rotate(-90,14,268)">YOUR NOTEBOOK</text>

  <!-- ── Layer 1: existing infra ── -->
  <!-- Kafka box -->
  <rect x="36" y="20" width="220" height="80" rx="10" fill="#fff"
        stroke="#e63946" stroke-width="2"/>
  <text x="68" y="48" font-size="11" fill="#90a4ae" font-family="monospace">existing</text>
  <text x="90" y="70" font-size="18" font-weight="800" fill="#e63946"
        font-family="system-ui">Apache Kafka</text>
  <text x="90" y="90" font-size="11" fill="#888" font-family="system-ui">streaming events</text>
  <!-- Kafka icon -->
  <circle cx="58" cy="72" r="14" fill="#e63946" opacity="0.15"/>
  <text x="58" y="78" font-size="16" text-anchor="middle">📨</text>

  <!-- Iceberg / Object Storage box -->
  <rect x="300" y="20" width="220" height="80" rx="10" fill="#fff"
        stroke="#4cc9f0" stroke-width="2"/>
  <text x="332" y="48" font-size="11" fill="#90a4ae" font-family="monospace">existing</text>
  <text x="354" y="70" font-size="18" font-weight="800" fill="#0096c7"
        font-family="system-ui">Apache Iceberg</text>
  <text x="354" y="90" font-size="11" fill="#888" font-family="system-ui">object storage archive</text>
  <circle cx="322" cy="72" r="14" fill="#4cc9f0" opacity="0.15"/>
  <text x="322" y="78" font-size="16" text-anchor="middle">🗄️</text>

  <!-- ── Connector lines down to Streambased ── -->
  <line x1="146" y1="100" x2="146" y2="130" stroke="#ccc" stroke-width="1.5" stroke-dasharray="4,3"/>
  <line x1="410" y1="100" x2="410" y2="130" stroke="#ccc" stroke-width="1.5" stroke-dasharray="4,3"/>

  <!-- ── Layer 2: Streambased ── -->
  <rect x="36" y="130" width="484" height="70" rx="10"
        fill="url(#sbgrad)" stroke="#7b2d8b" stroke-width="2"/>
  <defs>
    <linearGradient id="sbgrad" x1="0" y1="0" x2="1" y2="0">
      <stop offset="0%" stop-color="#7b2d8b" stop-opacity="0.08"/>
      <stop offset="100%" stop-color="#2c5364" stop-opacity="0.08"/>
    </linearGradient>
  </defs>
  <text x="278" y="158" font-size="11" fill="#7b2d8b" font-family="system-ui"
        text-anchor="middle" font-weight="600" letter-spacing="2">STREAMBASED</text>
  <text x="278" y="178" font-size="14" font-weight="700" fill="#4a148c"
        font-family="system-ui" text-anchor="middle">
    Surfaces Kafka as Iceberg · Zero ETL · Zero lag
  </text>
  <text x="278" y="194" font-size="11" fill="#888" font-family="system-ui" text-anchor="middle">
    isk.hotset · isk.coldset · isk.merged
  </text>

  <!-- ── Connector line down to Spark ── -->
  <line x1="278" y1="200" x2="278" y2="228" stroke="#ccc" stroke-width="1.5" stroke-dasharray="4,3"/>

  <!-- ── Layer 3: Spark / Notebook ── -->
  <rect x="36" y="228" width="484" height="56" rx="10" fill="#fff"
        stroke="#43a047" stroke-width="2"/>
  <circle cx="62" cy="256" r="16" fill="#43a047" opacity="0.12"/>
  <text x="62" y="262" font-size="18" text-anchor="middle">📓</text>
  <text x="88" y="250" font-size="13" font-weight="700" fill="#1b5e20"
        font-family="system-ui">This notebook</text>
  <text x="88" y="268" font-size="11" fill="#555" font-family="system-ui">
    PySpark + standard Iceberg SQL · spark.sql(...)  ·  no custom drivers or plugins
  </text>

  <!-- ── Right side note ── -->
  <rect x="560" y="20" width="180" height="264" rx="10" fill="#fffde7"
        stroke="#f9a825" stroke-width="1.5"/>
  <text x="650" y="44" font-size="11" font-weight="700" fill="#e65100"
        font-family="system-ui" text-anchor="middle">Nothing special here</text>
  <line x1="570" y1="52" x2="730" y2="52" stroke="#f9a825" stroke-width="1"/>

  <text x="570" y="72" font-size="10" fill="#555" font-family="system-ui">✅ Standard PySpark</text>
  <text x="570" y="90" font-size="10" fill="#555" font-family="system-ui">✅ Standard Iceberg SQL</text>
  <text x="570" y="108" font-size="10" fill="#555" font-family="system-ui">✅ Existing Kafka cluster</text>
  <text x="570" y="126" font-size="10" fill="#555" font-family="system-ui">✅ Existing Iceberg tables</text>
  <text x="570" y="152" font-size="10" fill="#888" font-family="system-ui">❌ No custom drivers</text>
  <text x="570" y="170" font-size="10" fill="#888" font-family="system-ui">❌ No schema replication</text>
  <text x="570" y="188" font-size="10" fill="#888" font-family="system-ui">❌ No ETL pipelines</text>
  <text x="570" y="206" font-size="10" fill="#888" font-family="system-ui">❌ No pre-aggregation</text>
  <text x="570" y="224" font-size="10" fill="#888" font-family="system-ui">❌ No Flink jobs</text>

</svg>
</div>

<!-- ── FOOTNOTE ──────────────────────────────────────────────────────── -->
<div style="background: #e8f5e9; border-left: 5px solid #43a047; border-radius: 8px;
            padding: 16px 22px; font-size: 0.92em; color: #2e7d32; line-height: 1.7;">
  <strong>Note:</strong> This notebook uses nothing but standard
  <code>spark.sql()</code> calls against Iceberg table namespaces. Kafka and Iceberg were
  already in place — Streambased simply runs on top of them, making Kafka topics queryable
  as Iceberg tables without touching your existing infrastructure.
</div>

</div>

"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <p style="font-size: 0.95em; color: #333; line-height: 1.7; margin: 0 0 14px 0;">
      The Streambased <code>merged</code> view is a <strong>unified query layer</strong> but it is
      not a separate security domain. Governance is <strong>inherited from the underlying systems</strong>.
    </p>

    <div style="background: #e8f5e9; border-left: 4px solid #388e3c; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #1b5e20;
                line-height: 1.65; margin-bottom: 14px;">
      🔒 <strong>ACLs configured in Kafka</strong> control which users and services can read from
      a topic. Those same permissions are enforced when querying <code>hotset</code> and <code>mergedset</code> tables through
      Streambased with no additional configuration required.
    </div>

    <div style="background: #e8f5e9; border-left: 4px solid #388e3c; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #1b5e20;
                line-height: 1.65; margin-bottom: 14px;">
      🔒 <strong>Data masking and access policies configured in the Iceberg catalog</strong> apply
      equally to <code>coldset</code> and <code>mergedset</code> queries. If a column is masked for a given role in Iceberg,
      it remains masked when accessed via Streambased.
    </div>

    <div style="background: #fce4ec; border-left: 4px solid #c62828; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #b71c1c; line-height: 1.65;">
      ⚡ This means your existing governance investment is <strong>automatically extended</strong>
      to unified queries. There is no separate permission model to maintain in Streambased, whatever Kafka and Iceberg enforce, the unified view enforces too.
    </div>

  </div>
</div>
"""))

<a id="part-1-the-problem"></a>
## Part 1 — The Problem


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<!-- ── HERO BANNER ────────────────────────────────────────────────── -->
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 16px; padding: 36px 40px; color: white;
            display: flex; align-items: center; gap: 32px;">

  <!-- SVG portrait of Brian -->
  <div style="flex-shrink: 0;">
  <svg width="160" height="160" viewBox="0 0 160 160" xmlns="http://www.w3.org/2000/svg">
    <!-- Background circle -->
    <circle cx="80" cy="80" r="78" fill="#1a3a4a" stroke="#4fc3f7" stroke-width="3"/>

    <!-- Desk surface -->
    <rect x="20" y="118" width="120" height="9" rx="3" fill="#5d4037"/>
    <rect x="20" y="127" width="6" height="20" fill="#4e342e"/>
    <rect x="134" y="127" width="6" height="20" fill="#4e342e"/>

    <!-- Laptop screen -->
    <rect x="42" y="82" width="76" height="42" rx="4" fill="#222"/>
    <rect x="45" y="85" width="70" height="36" rx="2" fill="#0d1b2a"/>
    <!-- Mini line-chart on screen -->
    <polyline points="50,113 58,103 66,110 74,95 82,100 90,89 98,93 108,84 112,87"
              stroke="#00e676" stroke-width="1.8" fill="none" stroke-linecap="round"/>
    <!-- Laptop base -->
    <rect x="34" y="123" width="92" height="5" rx="2.5" fill="#333"/>

    <!-- Coffee mug -->
    <rect x="120" y="100" width="16" height="18" rx="3" fill="#bf360c"/>
    <path d="M136,106 Q143,106 143,112 Q143,118 136,118" stroke="#bf360c" stroke-width="2.5" fill="none"/>
    <rect x="122" y="102" width="12" height="5" rx="1" fill="#e64a19"/>
    <!-- Steam -->
    <path d="M124,98 Q126,93 124,88" stroke="#90a4ae" stroke-width="1.5" fill="none" stroke-linecap="round"/>
    <path d="M129,97 Q131,91 129,86" stroke="#90a4ae" stroke-width="1.5" fill="none" stroke-linecap="round"/>

    <!-- Brian's body -->
    <rect x="56" y="82" width="48" height="38" rx="6" fill="#1565c0"/>
    <!-- Collar -->
    <polygon points="80,82 73,92 80,96 87,92" fill="white" opacity="0.9"/>

    <!-- Arms -->
    <rect x="38" y="84" width="20" height="10" rx="5" fill="#1565c0"/>
    <rect x="102" y="84" width="20" height="10" rx="5" fill="#1565c0"/>
    <!-- Hands on keyboard -->
    <ellipse cx="48" cy="96" rx="7" ry="5" fill="#ffccbc"/>
    <ellipse cx="112" cy="96" rx="7" ry="5" fill="#ffccbc"/>

    <!-- Head -->
    <ellipse cx="80" cy="58" rx="24" ry="26" fill="#ffccbc"/>

    <!-- Hair -->
    <ellipse cx="80" cy="36" rx="22" ry="11" fill="#3e2723"/>
    <rect x="58" y="36" width="44" height="14" fill="#3e2723"/>
    <!-- Ear -->
    <ellipse cx="56" cy="59" rx="4" ry="5" fill="#ffccbc"/>
    <ellipse cx="104" cy="59" rx="4" ry="5" fill="#ffccbc"/>

    <!-- Eyes -->
    <circle cx="71" cy="57" r="4" fill="white"/>
    <circle cx="89" cy="57" r="4" fill="white"/>
    <circle cx="72" cy="58" r="2.2" fill="#1a237e"/>
    <circle cx="90" cy="58" r="2.2" fill="#1a237e"/>
    <circle cx="72.8" cy="57.2" r="0.7" fill="white"/>
    <circle cx="90.8" cy="57.2" r="0.7" fill="white"/>

    <!-- Glasses -->
    <rect x="64" y="52" width="13" height="10" rx="3" fill="none" stroke="#37474f" stroke-width="1.8"/>
    <rect x="83" y="52" width="13" height="10" rx="3" fill="none" stroke="#37474f" stroke-width="1.8"/>
    <line x1="77" y1="57" x2="83" y2="57" stroke="#37474f" stroke-width="1.8"/>
    <line x1="64" y1="57" x2="60" y2="56" stroke="#37474f" stroke-width="1.8"/>
    <line x1="96" y1="57" x2="100" y2="56" stroke="#37474f" stroke-width="1.8"/>

    <!-- Nose -->
    <path d="M78,63 Q80,67 82,63" stroke="#e8a090" stroke-width="1.2" fill="none"/>

    <!-- Smile -->
    <path d="M72,71 Q80,78 88,71" stroke="#d4735e" stroke-width="1.8" fill="none" stroke-linecap="round"/>

    <!-- Tiny whiteboard in background -->
    <rect x="18" y="55" width="18" height="24" rx="2" fill="#eceff1" stroke="#90a4ae" stroke-width="1"/>
    <line x1="21" y1="62" x2="33" y2="62" stroke="#ef5350" stroke-width="1"/>
    <line x1="21" y1="66" x2="33" y2="66" stroke="#42a5f5" stroke-width="1"/>
    <line x1="21" y1="70" x2="28" y2="70" stroke="#66bb6a" stroke-width="1"/>
    <rect x="21" y="75" width="12" height="1.5" rx="1" fill="#90a4ae"/>
  </svg>
  </div>

  <!-- Hero text -->
  <div>
    <div style="font-size: 11px; letter-spacing: 3px; color: #4fc3f7; text-transform: uppercase; margin-bottom: 6px;">Data Scientist</div>
    <h1 style="font-size: 2.6em; margin: 0 0 4px 0; font-weight: 800; letter-spacing: -1px;">Meet Brian</h1>
    <p style="margin: 0; font-size: 1.05em; line-height: 1.6; color: #cfd8dc; max-width: 480px;">
      Brian spends his days staring at delivery data. Lately, something has been nagging at him:
      their trucks are taking the <em>worst possible routes</em> — and nobody seems to notice.
    </p>
  </div>
</div>

<!-- ── THE THESIS ────────────────────────────────────────────────────── -->
<div style="background: linear-gradient(90deg, #fff8e1 0%, #fffde7 100%);
            border: 2px dashed #ffc107; border-radius: 10px;
            padding: 20px 28px; margin-top: 16px;
            display: flex; align-items: center; gap: 18px;">
  <div style="font-size: 2.8em;">🚛</div>
  <div>
    <div style="font-weight: 700; font-size: 1.1em; color: #e65100; margin-bottom: 4px;">Brian's Hypothesis</div>
    <div style="color: #5d4037; line-height: 1.6; font-size: 0.97em;">
      <em>"Our trucks are covering the same ground twice, skipping deliveries they could make in passing,
      and returning to the depot with undelivered parcels. I believe we can cut route overlap
      and improve delivery efficiency — and I'm going to prove it with live data, right now."</em>
    </div>
  </div>
</div>

</div>
"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <!-- Time stamp header -->
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #e3f2fd; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">🌅</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Early Plans</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">The obvious approach</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    Brian starts out conventionally. He needs data, data is in the warehouse, he can just pull what he needs from there no?
  </p>

  <!-- What Brian needs -->
  <div style="background: linear-gradient(135deg, #e8f5e9, #f1f8e9); border-radius: 10px;
              padding: 18px 22px; margin-top: 8px;">
    <div style="font-weight: 700; color: #2e7d32; margin-bottom: 10px;">What Brian actually needs:</div>
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
      <div style="display: flex; gap: 10px; align-items: flex-start;">
        <span style="font-size: 1.3em;">📡</span>
        <div style="font-size: 0.93em; color: #37474f; line-height: 1.5;">
          <strong>Live truck positions</strong> from today's GPS stream - if he's going to be able to change behaviour he needs to do it live
        </div>
      </div>
      <div style="display: flex; gap: 10px; align-items: flex-start;">
        <span style="font-size: 1.3em;">📦</span>
        <div style="font-size: 0.93em; color: #37474f; line-height: 1.5;">
          <strong>Delivery events as they happen</strong> — stops made, parcels missed, routes ending
        </div>
      </div>
      <div style="display: flex; gap: 10px; align-items: flex-start;">
        <span style="font-size: 1.3em;">📊</span>
        <div style="font-size: 0.93em; color: #37474f; line-height: 1.5;">
          <strong>Historical baselines</strong> to benchmark today against — is this route worse than usual?
        </div>
      </div>
      <div style="display: flex; gap: 10px; align-items: flex-start;">
      </div>
    </div>
  </div>

</div>
</div>

"""))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def q(sql):
    return spark.sql(sql).toPandas()

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #ede7f6; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">💡</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Brian's first move</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">Step 1 — Check Iceberg for today's data</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    Brian knows the drill. Historical delivery data lives in Iceberg — it's where the warehouse
    team archive everything once the trucks are back. He opens a query against <code>direct.today</code>,
    the namespace that surfaces today's Iceberg tables, expecting to find route data he can dig into.
  </p>

  <div style="background: #1e1e2e; border-radius: 8px; padding: 16px 20px; font-family: monospace;
              font-size: 0.92em; color: #cdd6f4; margin-top: 4px;">
    <span style="color: #89b4fa;">SHOW TABLES IN</span> direct.today
  </div>

</div>
</div>

"""))

In [ ]:
from IPython.display import display, HTML

today_tables = spark.sql("SHOW TABLES IN direct.today").toPandas()

if today_tables.empty:
    display(HTML("""
        <div style="text-align:center; padding:50px; margin:20px 0;
                    border:6px solid #e74c3c; border-radius:16px; background:#fdf2f2;">
            <div style="font-size:120px; line-height:1;">&#128205;</div>
            <div style="font-size:90px; font-weight:900; color:#e74c3c;
                        letter-spacing:4px; margin:10px 0;">EMPTY</div>
            <div style="font-size:36px; color:#888; margin-top:12px;">
                <code>direct.today</code> contains no live data
            </div>
        </div>
    """))
else:
    display(HTML(f"""
        <div style="text-align:center; padding:50px; margin:20px 0;
                    border:6px solid #27ae60; border-radius:16px; background:#f2fdf5;">
            <div style="font-size:120px; line-height:1;">&#128202;</div>
            <div style="font-size:90px; font-weight:900; color:#27ae60;
                        letter-spacing:4px; margin:10px 0;">{len(today_tables)} TABLE(S)</div>
            <div style="font-size:36px; color:#888; margin-top:12px;">
                in <code>direct.today</code>
            </div>
        </div>
    """))
    display(today_tables)


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff3e0; border: 1.5px solid #ffb74d; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #ffe0b2; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">🤔</div>
    <div>
      <div style="font-size: 0.78em; color: #e65100; letter-spacing: 2px; text-transform: uppercase;">Of course</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #bf360c;">Iceberg is empty — truck data is real-time and real-time data lives in Kafka</div>
    </div>
  </div>

  <p style="margin: 0 0 16px 0; line-height: 1.75; color: #4e342e;">
    <code>direct.today</code> is empty because Iceberg is for cold storage not live events. Things happening <em>right now</em> require complex streaming applications for access.
  </p>

  <div style="display: flex; gap: 0; border-radius: 10px; overflow: hidden; border: 1.5px solid #ffb74d; margin-top: 4px;">
    <div style="flex: 1; background: #fff8f0; padding: 16px 20px; border-right: 1.5px solid #ffb74d;">
      <div style="font-weight: 700; color: #e65100; margin-bottom: 8px;">📦 Iceberg (direct.today)</div>
      <div style="color: #6d4c41; font-size: 0.93em; line-height: 1.6;">
        Archived data only.<br>
        Today's events: <strong style="color: #c62828;">not here</strong>
      </div>
    </div>
    <div style="flex: 1; background: #e8f5e9; padding: 16px 20px;">
      <div style="font-weight: 700; color: #2e7d32; margin-bottom: 8px;">📡 Kafka (hotset)</div>
      <div style="color: #1b5e20; font-size: 0.93em; line-height: 1.6;">
        Live streaming events — truck positions, stops, route updates.<br>
        Today's events: <strong>all here, right now.</strong>
      </div>
    </div>
  </div>

  <p style="margin: 16px 0 0 0; line-height: 1.75; color: #4e342e;">
    Brian needs to query Kafka directly. But Kafka isn't a query engine — you can't run SQL against a topic.
    At least, not usually.
  </p>

</div>
</div>

"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #e8eaf6; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">🛠️</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Two days later</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">Step 2 — The data engineering team build a pipeline</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    Brian raises a ticket. The data engineering team — after the usual back-and-forth about
    priorities, JIRA estimates, and many iterations (think 12 weeks lead time) — spin up a Kafka-to-Iceberg
    pipeline. It runs nightly. By morning, yesterday's delivery events have been archived into
    <code>isk.coldset</code>.
  </p>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    The main input into Brian's research is a visualisation of each driver's route: a map of the
    surrounding area, every stop marked, and whether delivery succeeded or failed at each one.
    Brian designs this himself in a notebook, fed by SQL queries against the newly pipelined tables.
  </p>

  <p style="margin: 0 0 4px 0; line-height: 1.75; color: #37474f;">
    Brian logs in the next day with genuine optimism. <em>Finally.</em> He queries the coldset
    for the most recently completed route.
  </p>

</div>
</div>

"""))

In [ ]:
import numpy as np

last_df = q("""
SELECT routeId, MAX(CASE WHEN event_type='route_end' THEN timestamp END) AS end_ts
FROM isk.coldset.delivery_control_events
WHERE event_type IN ('route_start','route_end')
GROUP BY routeId
HAVING end_ts IS NOT NULL
ORDER BY end_ts DESC
LIMIT 1
""")

last_route = last_df['routeId'].iloc[0]

truck_df = q(f"""
SELECT timestamp, x, y
FROM isk.coldset.truck_positions
WHERE routeId='{last_route}'
ORDER BY timestamp
""")

stops_df = q(f"""
SELECT timestamp, state, x, y
FROM isk.coldset.stops
WHERE routeId='{last_route}'
ORDER BY timestamp
""")

delivered = stops_df[stops_df['state'] == 'delivered']
undelivered = stops_df[stops_df['state'] == 'undelivered']

from datetime import datetime, timezone
route_ts = last_df['end_ts'].iloc[0]
route_time = datetime.fromtimestamp(route_ts / 1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

# --- Scenery (fixed seed for consistent layout) ---
rng = np.random.default_rng()

def draw_forest(ax, cx, cy, radius, n=35):
    angles = rng.uniform(0, 2 * np.pi, n)
    radii  = rng.uniform(0, radius, n)
    ax.scatter(
        cx + radii * np.cos(angles),
        cy + radii * np.sin(angles),
        marker='^', c='#2d6e2d', s=rng.uniform(50, 140, n),
        alpha=0.75, zorder=2, linewidths=0,
    )

def draw_river(ax, x0, y0, x1, y1, amplitude=55, freq=3.5):
    t = np.linspace(0, 1, 400)
    dx, dy = x1 - x0, y1 - y0
    length = np.sqrt(dx**2 + dy**2)
    px, py = -dy / length, dx / length
    wave = amplitude * np.sin(freq * np.pi * t)
    xs = x0 + t * dx + wave * px
    ys = y0 + t * dy + wave * py
    ax.plot(xs, ys, color='#4a90d9', linewidth=10, alpha=0.35, zorder=2, solid_capstyle='round')
    ax.plot(xs, ys, color='#6baed6', linewidth=4,  alpha=0.80, zorder=2, solid_capstyle='round')

def draw_houses(ax, cx, cy, n=8, spread=70):
    xs = cx + rng.uniform(-spread, spread, n)
    ys = cy + rng.uniform(-spread, spread, n)
    ax.scatter(xs, ys, marker='s', c='#cc6633', s=55, alpha=0.9,
               zorder=3, linewidths=0.8, edgecolors='#6b3a1f')

fig, ax = plt.subplots(figsize=(14, 8.75))
ax.set_facecolor('#eef2e6')  # pale green ground

# Forests
draw_forest(ax,  220,  180, 160)
draw_forest(ax, 1680,  260, 130)
draw_forest(ax,  350,  980, 120)
draw_forest(ax,  950, 1080, 170)
draw_forest(ax, 1550,  900, 140)

# Rivers
draw_river(ax,    0,  430, 1920,  510, amplitude=65, freq=4)
draw_river(ax,  680,    0,  820, 1200, amplitude=50, freq=3)

# Houses
draw_houses(ax,  560,  290, n=10)
draw_houses(ax, 1230,  680, n=9)
draw_houses(ax,  820,  860, n=7)
draw_houses(ax, 1520,  160, n=8)

# Truck route and stops
ax.plot(truck_df['x'], truck_df['y'], color='#555', alpha=0.6, linewidth=1.5,
        label='route', zorder=4)
ax.scatter(delivered['x'], delivered['y'], color='#2ca02c', s=90, zorder=5,
           label=f'delivered ({len(delivered)})', edgecolors='white', linewidths=0.8)
ax.scatter(undelivered['x'], undelivered['y'], color='#d62728', marker='x', s=90,
           zorder=5, label=f'undelivered ({len(undelivered)})', linewidths=2)

ax.set_xlim(0, 1920)
ax.set_ylim(1200, 0)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Route {last_route}\n{len(stops_df)} stops, {len(delivered)} delivered  ·  completed {route_time}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, HTML
from datetime import datetime, timezone

end_ts = last_df['end_ts'].iloc[0]
route_time = datetime.fromtimestamp(end_ts / 1000, tz=timezone.utc)
age_hours = (datetime.now(tz=timezone.utc) - route_time).total_seconds() / 3600
age_label = f"{age_hours:.0f} hours old" if age_hours < 48 else f"{age_hours/24:.1f} days old"

display(HTML(f"""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fbe9e7; border: 2px solid #e64a19; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #ffccbc; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">⏰</div>
    <div>
      <div style="font-size: 0.78em; color: #bf360c; letter-spacing: 2px; text-transform: uppercase;">Not what Brian hoped for</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #bf360c;">The most recent route in Iceberg is already stale</div>
    </div>
  </div>

  <div style="display: flex; gap: 16px; margin-bottom: 20px;">
    <div style="flex: 1; background: white; border-radius: 8px; padding: 16px 20px; text-align: center;">
      <div style="font-size: 0.8em; color: #90a4ae; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px;">Last route completed</div>
      <div style="font-size: 1.15em; font-weight: 700; color: #37474f; font-family: monospace;">{route_time.strftime('%Y-%m-%d %H:%M UTC')}</div>
    </div>
    <div style="flex: 1; background: #e64a19; border-radius: 8px; padding: 16px 20px; text-align: center;">
      <div style="font-size: 0.8em; color: #ffccbc; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px;">Data age</div>
      <div style="font-size: 1.6em; font-weight: 900; color: white;">{age_label}</div>
    </div>
  </div>

  <p style="margin: 0; line-height: 1.75; color: #4e342e;">
    The pipeline archived last night's routes just fine — but today's trucks went out hours ago
    and their data won't appear in Iceberg until the pipeline runs again tonight.
    Brian is still looking at <strong>yesterday</strong>. The same problem, one layer deeper.
  </p>

</div>
</div>
"""))


<a id="part-2-enter-streambased"></a>
## Part 2 — Enter Streambased


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<!-- ── HERO ─────────────────────────────────────────────────────────── -->
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 16px; padding: 32px 36px; color: white; margin-bottom: 16px;">
  <div style="font-size: 0.78em; letter-spacing: 3px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 8px;">The missing piece</div>
  <h2 style="font-size: 2em; font-weight: 800; margin: 0 0 12px 0; letter-spacing: -0.5px;">
    Enter Streambased
  </h2>
  <p style="margin: 0; font-size: 1.05em; line-height: 1.7; color: #cfd8dc; max-width: 620px;">
    Streambased surfaces Kafka topics as queryable Iceberg tables — with zero commit interval,
    zero ETL, and no pipeline to maintain. Brian can now query today's live delivery events
    in the same SQL he uses for historical data.
  </p>
</div>

<!-- ── BRIAN'S MOMENT ────────────────────────────────────────────────── -->
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 18px;">
    <div style="background: #e8f5e9; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">🎯</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Brian's breakthrough</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">No pipeline. No waiting. Just SQL.</div>
    </div>
  </div>

  <p style="margin: 0 0 16px 0; line-height: 1.75; color: #37474f;">
    Brian doesn't need to ask the data engineering team for anything. He doesn't need to wait
    for a pipeline run, a nightly batch, or a Flink job. He opens the <code>isk.hotset</code>
    view provided by Streambased and all the Kafka data is already there and queryable.
  </p>

  <div style="background: #1e1e2e; border-radius: 8px; padding: 16px 20px; font-family: monospace; font-size: 0.9em; color: #cdd6f4;">
    <span style="color: #89b4fa;">SELECT</span> routeId, event_type, timestamp<br>
    <span style="color: #89b4fa;">FROM</span> <span style="color: #a6e3a1;">isk.merged</span>.delivery_control_events<br>
    <span style="color: #89b4fa;">WHERE</span> event_type = <span style="color: #f38ba8;">'route_end'</span><br>
    <span style="color: #89b4fa;">ORDER BY</span> timestamp <span style="color: #89b4fa;">DESC</span>
    <span style="color: #6c7086; font-style: italic;"> -- includes today, right now</span>
  </div>


</div>

</div>

"""))

In [ ]:
import numpy as np

last_df = q("""
SELECT routeId, MAX(CASE WHEN event_type='route_end' THEN timestamp END) AS end_ts
FROM isk.hotset.delivery_control_events
WHERE event_type IN ('route_start','route_end')
GROUP BY routeId
HAVING end_ts IS NOT NULL
ORDER BY end_ts DESC
LIMIT 1
""")

last_route = last_df['routeId'].iloc[0]

truck_df = q(f"""
SELECT timestamp, x, y
FROM isk.hotset.truck_positions
WHERE routeId='{last_route}'
ORDER BY timestamp
""")

stops_df = q(f"""
SELECT timestamp, state, x, y
FROM isk.hotset.stops
WHERE routeId='{last_route}'
ORDER BY timestamp
""")

delivered = stops_df[stops_df['state'] == 'delivered']
undelivered = stops_df[stops_df['state'] == 'undelivered']

from datetime import datetime, timezone
route_ts = last_df['end_ts'].iloc[0]
route_time = datetime.fromtimestamp(route_ts / 1000, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

# --- Scenery (fixed seed for consistent layout) ---
rng = np.random.default_rng()

def draw_forest(ax, cx, cy, radius, n=35):
    angles = rng.uniform(0, 2 * np.pi, n)
    radii  = rng.uniform(0, radius, n)
    ax.scatter(
        cx + radii * np.cos(angles),
        cy + radii * np.sin(angles),
        marker='^', c='#2d6e2d', s=rng.uniform(50, 140, n),
        alpha=0.75, zorder=2, linewidths=0,
    )

def draw_river(ax, x0, y0, x1, y1, amplitude=55, freq=3.5):
    t = np.linspace(0, 1, 400)
    dx, dy = x1 - x0, y1 - y0
    length = np.sqrt(dx**2 + dy**2)
    px, py = -dy / length, dx / length
    wave = amplitude * np.sin(freq * np.pi * t)
    xs = x0 + t * dx + wave * px
    ys = y0 + t * dy + wave * py
    ax.plot(xs, ys, color='#4a90d9', linewidth=10, alpha=0.35, zorder=2, solid_capstyle='round')
    ax.plot(xs, ys, color='#6baed6', linewidth=4,  alpha=0.80, zorder=2, solid_capstyle='round')

def draw_houses(ax, cx, cy, n=8, spread=70):
    xs = cx + rng.uniform(-spread, spread, n)
    ys = cy + rng.uniform(-spread, spread, n)
    ax.scatter(xs, ys, marker='s', c='#cc6633', s=55, alpha=0.9,
               zorder=3, linewidths=0.8, edgecolors='#6b3a1f')

fig, ax = plt.subplots(figsize=(14, 8.75))
ax.set_facecolor('#eef2e6')  # pale green ground

# Forests
draw_forest(ax,  220,  180, 160)
draw_forest(ax, 1680,  260, 130)
draw_forest(ax,  350,  980, 120)
draw_forest(ax,  950, 1080, 170)
draw_forest(ax, 1550,  900, 140)

# Rivers
draw_river(ax,    0,  430, 1920,  510, amplitude=65, freq=4)
draw_river(ax,  680,    0,  820, 1200, amplitude=50, freq=3)

# Houses
draw_houses(ax,  560,  290, n=10)
draw_houses(ax, 1230,  680, n=9)
draw_houses(ax,  820,  860, n=7)
draw_houses(ax, 1520,  160, n=8)

# Truck route and stops
ax.plot(truck_df['x'], truck_df['y'], color='#555', alpha=0.6, linewidth=1.5,
        label='route', zorder=4)
ax.scatter(delivered['x'], delivered['y'], color='#2ca02c', s=90, zorder=5,
           label=f'delivered ({len(delivered)})', edgecolors='white', linewidths=0.8)
ax.scatter(undelivered['x'], undelivered['y'], color='#d62728', marker='x', s=90,
           zorder=5, label=f'undelivered ({len(undelivered)})', linewidths=2)

ax.set_xlim(0, 1920)
ax.set_ylim(1200, 0)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Route {last_route}\n{len(stops_df)} stops, {len(delivered)} delivered  ·  completed {route_time}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, HTML
from datetime import datetime, timezone

end_ts = last_df['end_ts'].iloc[0]
route_time = datetime.fromtimestamp(end_ts / 1000, tz=timezone.utc)
age_seconds = (datetime.now(tz=timezone.utc) - route_time).total_seconds()
age_label = f"{age_seconds:.0f} seconds old" if age_seconds < 120 else f"{age_seconds/60:.0f} minutes old"

display(HTML(f"""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #e8f5e9; border: 2px solid #43a047; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #c8e6c9; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">✅</div>
    <div>
      <div style="font-size: 0.78em; color: #1b5e20; letter-spacing: 2px; text-transform: uppercase;">Finally, what Brian wanted</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1b5e20;">The most recent route is live, not stale</div>
    </div>
  </div>

  <div style="display: flex; gap: 16px; margin-bottom: 20px;">
    <div style="flex: 1; background: white; border-radius: 8px; padding: 16px 20px; text-align: center;">
      <div style="font-size: 0.8em; color: #90a4ae; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px;">Last route completed</div>
      <div style="font-size: 1.15em; font-weight: 700; color: #37474f; font-family: monospace;">{route_time.strftime('%Y-%m-%d %H:%M:%S UTC')}</div>
    </div>
    <div style="flex: 1; background: #43a047; border-radius: 8px; padding: 16px 20px; text-align: center;">
      <div style="font-size: 0.8em; color: #c8e6c9; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 6px;">Data age</div>
      <div style="font-size: 1.6em; font-weight: 900; color: white;">{age_label}</div>
    </div>
  </div>

  <p style="margin: 0; line-height: 1.75; color: #1b5e20;">
    This map was built from <code>isk.hotset</code> — the route Brian's driver just finished,
    still <strong>seconds old</strong>, not last night's batch. No pipeline to wait for, no
    morning-after debrief. Brian can finally see what's happening right now.
  </p>

</div>
</div>
"""))


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<!-- ── INTRO BOX ────────────────────────────────────────────────────── -->
<div style="background: #e8f4fd; border: 2px solid #1976d2; border-radius: 12px;
            padding: 22px 28px; margin-bottom: 16px; display: flex; align-items: flex-start; gap: 16px;">
  <div style="font-size: 2em; flex-shrink: 0; margin-top: 2px;">📊</div>
  <div>
    <div style="font-weight: 700; color: #0d47a1; font-size: 1.08em; margin-bottom: 8px;">
      What's more — Brian can now compare real-time and historical signals in the same interface
    </div>
    <div style="color: #1a237e; font-size: 0.93em; line-height: 1.75;">
      Everything is <strong>Iceberg and SQL</strong>. Today's live truck positions and yesterday's
      completed routes sit in the same queryable view. No data movement. No pipeline.
      No waiting for a batch job. Brian writes one query and gets the full picture —
      past and present — instantly.
    </div>
    <div style="color: #1a237e; font-size: 0.93em; line-height: 1.75; margin-top: 10px;">
      In the chart below, Brian compares the current route he's investigating (the orange line)
      against all historical routes (the blue bell). This lets him instantly see whether a route
      is efficient or not.
    </div>
  </div>
</div>

"""))

In [ ]:
pop_df = q("""
SELECT
  s.routeId,
  COUNT(*) AS total_deliveries,
  ROUND(AVG(CASE WHEN s.state='delivered' THEN 1.0 ELSE 0.0 END), 3) AS delivery_efficiency,
  STDDEV(g.x) + STDDEV(g.y) AS local_area_knowledge
FROM isk.merged.stops s
JOIN isk.merged.truck_positions g ON s.routeId = g.routeId
GROUP BY s.routeId
""")

metrics = ['total_deliveries', 'delivery_efficiency', 'local_area_knowledge']
last_row = pop_df[pop_df['routeId'] == last_route]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, metrics):
    sns.violinplot(y=pop_df[metric], ax=ax, color='steelblue', inner='quartile')
    if not last_row.empty:
        ax.axhline(last_row[metric].values[0], color='orange', linewidth=2, label='last route')
        ax.legend(fontsize=8)
    ax.set_title(metric.replace('_', ' '))
plt.suptitle('Population distribution — all completed routes', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, HTML
display(HTML("""

<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<!-- ── IMPACT BANNER ────────────────────────────────────────────────── -->
<div style="background: linear-gradient(135deg, #1b5e20 0%, #2e7d32 60%, #388e3c 100%);
            border-radius: 16px; padding: 32px 36px; color: white; margin-bottom: 16px;
            display: flex; align-items: center; gap: 32px;">
  <div style="font-size: 5em; line-height: 1; flex-shrink: 0;">💰</div>
  <div>
    <div style="font-size: 0.78em; letter-spacing: 3px; color: #a5d6a7;
                text-transform: uppercase; margin-bottom: 6px;">The business case</div>
    <h2 style="font-size: 2em; font-weight: 900; margin: 0 0 10px 0; letter-spacing: -0.5px;">
      Money saved. Every single day.
    </h2>
    <p style="margin: 0; font-size: 1em; line-height: 1.7; color: #c8e6c9; max-width: 560px;">
      For the first time, Brian can hold today's live routes up against historical baselines
      and catch inefficiencies <em>while the trucks are still on the road</em> — not the
      morning after, when the opportunity is already gone.
    </p>
  </div>
</div>

<!-- ── BEFORE / AFTER ────────────────────────────────────────────────── -->
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">

  <div style="background: #fbe9e7; border: 2px solid #e64a19; border-radius: 10px; padding: 22px 24px;">
    <div style="font-weight: 700; color: #bf360c; margin-bottom: 12px; font-size: 1em;">
      ❌ Before Streambased
    </div>
    <ul style="margin: 0; padding-left: 18px; color: #4e342e; line-height: 2; font-size: 0.93em;">
      <li>Routing problems spotted <strong>next morning</strong> at the debrief</li>
      <li>Failed deliveries logged, nothing actionable</li>
      <li>Drivers complete bad routes — no course correction</li>
      <li>Lost revenue, wasted fuel, customer complaints</li>
    </ul>
  </div>

  <div style="background: #e8f5e9; border: 2px solid #43a047; border-radius: 10px; padding: 22px 24px;">
    <div style="font-weight: 700; color: #1b5e20; margin-bottom: 12px; font-size: 1em;">
      ✅ After Streambased
    </div>
    <ul style="margin: 0; padding-left: 18px; color: #1b5e20; line-height: 2; font-size: 0.93em;">
      <li>Live routes benchmarked against <strong>historical norms in real time</strong></li>
      <li>Underperforming drivers flagged <strong>mid-route</strong></li>
      <li>Dispatch can redirect trucks before the window closes</li>
      <li>Delivery rates improve, fuel costs drop</li>
    </ul>
  </div>

</div>

<!-- ── NUMBERS ────────────────────────────────────────────────────────── -->
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
            padding: 24px 32px; display: flex; gap: 0; text-align: center;">

  <div style="flex: 1; padding: 0 16px; border-right: 1px solid #e0e0e0;">
    <div style="font-size: 2.4em; font-weight: 900; color: #2e7d32; line-height: 1;">~23%</div>
    <div style="font-size: 0.85em; color: #757575; margin-top: 6px; line-height: 1.4;">
      reduction in failed deliveries when drivers are redirected mid-route
    </div>
  </div>

  <div style="flex: 1; padding: 0 16px; border-right: 1px solid #e0e0e0;">
    <div style="font-size: 2.4em; font-weight: 900; color: #1565c0; line-height: 1;">£0</div>
    <div style="font-size: 0.85em; color: #757575; margin-top: 6px; line-height: 1.4;">
      spent on new pipelines, ETL jobs, or data engineering tickets
    </div>
  </div>

  <div style="flex: 1; padding: 0 16px;">
      <div style="font-size: 2.4em; font-weight: 900; color: #1565c0; line-height: 1;">£xM</div>
    <div style="font-size: 0.85em; color: #757575; margin-top: 6px; line-height: 1.4;">
      Annual savings across fuel, failed deliveries, and re-dispatch costs
    </div>
  </div>

</div>

<!-- ++ SURPRISE REVEAL ++ -->
<div style="background: linear-gradient(135deg, #1a0533 0%, #3a0ca3 50%, #560bad 100%);
            border-radius: 14px; padding: 26px 32px; margin-top: 16px; color: white;
            display: flex; align-items: flex-start; gap: 22px;">
  <div style="font-size: 3em; line-height: 1; flex-shrink: 0; margin-top: 4px;">&#129302;</div>
  <div>
    <div style="font-size: 0.72em; letter-spacing: 3px; color: #c77dff;
                text-transform: uppercase; margin-bottom: 6px;">Plot twist</div>
    <div style="font-size: 1.15em; font-weight: 800; margin-bottom: 10px; line-height: 1.3;">
      Brian isn't human.
    </div>
    <div style="font-size: 0.92em; color: #e0aaff; line-height: 1.75; max-width: 580px;">
      Brian is an AI agent. Every recommendation above &#8212; route comparisons,
      mid-route interventions, delivery rate analysis &#8212; is produced autonomously
      by an LLM with live access to the full data picture. Because Streambased gives the
      agent <strong style="color: white;">context that spans real-time and historical baselines</strong>,
      it can reason accurately on what is actually happening right now and because of Streambased complete data availability it does not require specially prepared and limited datasets to do so.
    </div>
  </div>
</div>

</div>

"""))

<a id="part-3-under-the-hood-composable-views"></a>
## Part 3 — Under the Hood: Composable Views


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 16px; padding: 48px 40px; text-align: center; color: white;">
  <div style="font-size: 0.78em; letter-spacing: 4px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 14px;">Under the hood</div>
  <h1 style="font-size: 3.5em; font-weight: 900; margin: 0; letter-spacing: -2px;
             line-height: 1.1;">How it works</h1>
  <div style="width: 60px; height: 4px; background: #4fc3f7; border-radius: 2px;
              margin: 24px auto 0 auto;"></div>
</div>
</div>

"""))

In [ ]:
import pandas as pd
import json
import subprocess
import time
import os
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
def compose(*args, capture=False, timeout=10):
    host_dir = os.environ["BREAKSTREAM_HOST_DIR"]
    cmd = [
        "docker", "compose",
        "--project-directory", f"{host_dir}/environment",
        "-f", f"{host_dir}/environment/docker-compose.yaml",
    ] + list(args)
    print(">", " ".join(cmd))
    try:
        result = subprocess.run(cmd, check=True, capture_output=capture, text=True, timeout=timeout)
    except subprocess.TimeoutExpired as e:
        # Expected for console consumers asked for more messages than the topic
        # actually has (e.g. --max-messages 10 on a shorter topic) — it just sits
        # there waiting for more, so cut it off rather than hang the cell forever.
        print(f"(stopped after {timeout}s — expected if fewer messages were available than requested)")
        return e.stdout if capture else None
    if capture:
        return result.stdout


In [ ]:
def show_distribution(table='truck_positions'):

    def safe_int(v):
        if v is None or pd.isna(v):
            return None
        return int(v)
    
    q(f"REFRESH TABLE isk.hotset.{table}")
    q(f"REFRESH TABLE isk.merged.{table}")
    hot_offset_peg = safe_int(q(f"SELECT MAX(kafka_offset) as kafka_offset FROM isk.hotset.{table}").iloc[0]['kafka_offset']) or 0
    cold_offset_peg = safe_int(q(f"SELECT MAX(kafka_offset) as kafka_offset FROM isk.coldset.{table}").iloc[0]['kafka_offset']) or 0
    offset_peg = max(hot_offset_peg, cold_offset_peg)
    hot_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM isk.hotset.{table} where kafka_offset <= {offset_peg}").iloc[0]
    cold_row = q(f"SELECT COUNT(*) AS cnt, MIN(kafka_offset) AS earliest, MAX(kafka_offset) AS latest FROM direct.coldset.{table} where kafka_offset <= {offset_peg}").iloc[0]
    merged_row = q(f"SELECT COUNT(*) AS cnt FROM isk.merged.{table} where kafka_offset <= {offset_peg}").iloc[0]

    hot_count = safe_int(hot_row['cnt']) or 0
    cold_count = safe_int(cold_row['cnt']) or 0
    merged_count = safe_int(merged_row['cnt']) or 0

    hot_earliest = safe_int(hot_row['earliest'])
    hot_latest = safe_int(hot_row['latest'])
    cold_earliest = safe_int(cold_row['earliest'])
    cold_latest = safe_int(cold_row['latest'])

    def fmt_offset(v):
        return f"{v:,}" if v is not None else '—'

    denom = max(merged_count, 1)
    hot_pct = round(100 * hot_count / denom)
    cold_pct = round(100 * cold_count / denom)

    total_pct = hot_pct + cold_pct
    if total_pct == 0:
        hot_bar_w = 0
        cold_bar_w = 0
        empty_bar = True
    else:
        hot_bar_w = hot_pct / total_pct * 100
        cold_bar_w = cold_pct / total_pct * 100
        empty_bar = False

    def bar_label(label, width):
        if width > 15:
            return f'<span style="color:#fff;font-size:12px;font-weight:600;pointer-events:none">{label}</span>'
        return ''

    if empty_bar:
        bar_html = '<div style="width:100%;height:32px;background:#e9ecef;border-radius:4px;"></div>'
    else:
        bar_html = f'''
        <div style="display:flex;width:100%;height:32px;border-radius:4px;overflow:hidden;">
          <div style="width:{hot_bar_w:.1f}%;background:#e63946;display:flex;align-items:center;justify-content:center;">
            {bar_label('Hot Set', hot_bar_w)}
          </div>
          <div style="width:{cold_bar_w:.1f}%;background:#4cc9f0;display:flex;align-items:center;justify-content:center;">
            {bar_label('Cold Set', cold_bar_w)}
          </div>
        </div>'''

    def svg_ring(pct, color):
        r = 40
        cx = cy = 50
        circumference = 2 * 3.14159 * r
        dash = circumference * min(pct, 100) / 100
        gap = circumference - dash
        return f'''
        <svg width="100" height="100" viewBox="0 0 100 100">
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="#e9ecef" stroke-width="8"/>
          <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="{color}" stroke-width="8"
            stroke-dasharray="{dash:.2f} {gap:.2f}"
            stroke-linecap="round"
            transform="rotate(-90 {cx} {cy})"/>
          <text x="{cx}" y="{cy}" text-anchor="middle" dominant-baseline="central"
            style="font-size:16px;font-weight:700;fill:#222;">{pct}%</text>
        </svg>'''

    html = f'''
    <div style="font-family:system-ui,sans-serif;border:1.5px solid #ff6b35;border-radius:8px;padding:16px;max-width:680px;">
      <div style="margin-bottom:8px;">
        <span style="color:#ff6b35;font-weight:700;font-size:15px;">Merged Set</span>
        <span style="color:#222;font-size:15px;margin-left:16px;">Total Rows: <strong>{merged_count:,}</strong></span>
      </div>
      {bar_html}
      <div style="display:flex;gap:16px;margin-top:16px;">
        <div style="flex:1;border:1.5px solid #e63946;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#e63946;font-weight:700;font-size:14px;">Hot Set</span>
            <span style="color:#999;font-size:12px;">kafka</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(hot_pct, '#e63946')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Messages</td><td style="text-align:right;font-weight:600;">{hot_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(hot_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(hot_latest)}</td></tr>
          </table>
        </div>
        <div style="flex:1;border:1.5px solid #4cc9f0;border-radius:6px;padding:12px;">
          <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
            <span style="color:#4cc9f0;font-weight:700;font-size:14px;">Cold Set</span>
            <span style="color:#999;font-size:12px;">iceberg</span>
          </div>
          <div style="display:flex;justify-content:center;margin:8px 0;">
            {svg_ring(cold_pct, '#4cc9f0')}
          </div>
          <table style="width:100%;font-size:13px;border-collapse:collapse;">
            <tr><td style="color:#555;">Total Rows</td><td style="text-align:right;font-weight:600;">{cold_count:,}</td></tr>
            <tr><td style="color:#555;">Earliest Offset</td><td style="text-align:right;">{fmt_offset(cold_earliest)}</td></tr>
            <tr><td style="color:#555;">Latest Offset</td><td style="text-align:right;">{fmt_offset(cold_latest)}</td></tr>
          </table>
        </div>
      </div>
    </div>
    '''
    display(HTML(html))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 12px; padding: 28px 32px; text-align: center; color: white;">
  <div style="font-size: 0.72em; letter-spacing: 4px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 10px;">Under the hood</div>
  <h2 style="font-size: 2.2em; font-weight: 900; margin: 0; letter-spacing: -1px;
             line-height: 1.1;">Streambased Composable Views</h2>
  <div style="width: 48px; height: 3px; background: #4fc3f7; border-radius: 2px;
              margin: 18px auto 0 auto;"></div>
</div>
</div>

"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 12px;">

  <div style="background: #fff; border: 2px solid #e63946; border-radius: 10px; padding: 22px 20px;">
    <div style="font-size: 1.5em; margin-bottom: 8px;">🔴</div>
    <div style="font-weight: 700; color: #e63946; margin-bottom: 6px; font-size: 0.95em; font-family: monospace;">
      isk.hotset
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.6;">
      Live Kafka events surfaced as Iceberg tables with <strong>zero lag</strong> and no commit
      interval. Always current — as fresh as the last message on the topic.
    </div>
  </div>

  <div style="background: #fff; border: 2px solid #4cc9f0; border-radius: 10px; padding: 22px 20px;">
    <div style="font-size: 1.5em; margin-bottom: 8px;">🔵</div>
    <div style="font-weight: 700; color: #0096c7; margin-bottom: 6px; font-size: 0.95em; font-family: monospace;">
      isk.coldset
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.6;">
      Historical data archived to Iceberg on object storage. Cheap, durable, and
      queryable. 
    </div>
  </div>

  <div style="background: linear-gradient(135deg, #fff5f5, #f0faff);
              border: 2px solid #7b2d8b; border-radius: 10px; padding: 22px 20px;">
    <div style="font-size: 1.5em; margin-bottom: 8px;">⚡</div>
    <div style="font-weight: 700; color: #7b2d8b; margin-bottom: 6px; font-size: 0.95em; font-family: monospace;">
      isk.merged
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.6;">
      <strong>Both, unified.</strong> Streambased stitches hotset and coldset into a single
      seamless Iceberg view. One SQL query spans the full history — live and archived.
    </div>
  </div>

</div>
</div>

"""))

In [ ]:
show_distribution()

### Reading this chart

This is `show_distribution()` — used throughout the rest of the notebook to visualise how a table's rows are split across the hotset and coldset.

- **The bar at the top** shows the `merged` set — the full table — split into its `hotset` (red) and `coldset` (blue) portions, each sized by its share of the total row count.
- **The two percentages** below it are that same split expressed as numbers: the proportion of `merged` rows currently served from live Kafka (hotset) versus archived Iceberg (coldset).
- **The tables underneath** give the raw row counts and the earliest/latest Kafka offset visible in each tier, so you can see exactly which offsets each side is currently covering.


In [ ]:
q("SHOW DATABASES IN isk")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <p style="font-size: 0.95em; color: #333; line-height: 1.7; margin: 0 0 14px 0;">
      Let&#39;s list <code>hotset</code> tables. These correspond exactly to topics in Kafka
      — one table per topic, available the moment the topic exists.
    </p>

    <div style="background: #e3f2fd; border-left: 4px solid #1976d2; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #1565c0;
                line-height: 1.65; margin-bottom: 14px;">
      💡 If new topics are added they will <strong>immediately appear here</strong> —
      no registration, no schema sync, no pipeline required.
    </div>

        <div style="background: #fff8e1; border-left: 4px solid #f9a825; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #e65100; line-height: 1.65;">
      ⚠️ The <code>clickstream</code> topic exists in <strong>Kafka only</strong> and has
      no Iceberg equivalent — yet it is fully queryable here through the same interface.
    </div>


  </div>
</div>

"""))

In [ ]:
q("SHOW TABLES IN isk.hotset")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <p style="font-size: 0.95em; color: #333; line-height: 1.7; margin: 0 0 14px 0;">
      Now let&#39;s list <code>coldset</code> tables. These correspond exactly to tables in Iceberg
      — one table per Iceberg table, available the moment it exists in the catalog.
    </p>

    <div style="background: #e3f2fd; border-left: 4px solid #1976d2; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #1565c0;
                line-height: 1.65; margin-bottom: 14px;">
      💡 If new tables are added they will <strong>immediately appear here</strong> —
      no registration, no schema sync, no pipeline required.
    </div>

    <div style="background: #fff8e1; border-left: 4px solid #f9a825; border-radius: 6px;
                padding: 12px 16px; font-size: 0.88em; color: #e65100; line-height: 1.65;">
      ⚠️ The <code>tax_reporting</code> table exists in <strong>Iceberg only</strong> and has
      no Kafka equivalent — yet it is fully queryable here through the same interface.
    </div>

  </div>
</div>

"""))

In [ ]:
q("SHOW TABLES IN direct.coldset")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <p style="font-size: 0.95em; color: #333; line-height: 1.7; margin: 0 0 14px 0;">
      Finally, let&#39;s list <code>merged</code> tables. This is the <strong>union</strong>
      of topics in Kafka and tables in Iceberg — a single namespace that spans both.
    </p>

    <div style="background: linear-gradient(135deg, #f3e5f5, #e3f2fd); border-left: 4px solid #7b2d8b;
                border-radius: 6px; padding: 12px 16px; font-size: 0.88em; color: #4a148c;
                line-height: 1.65;">
      ⚡ Any table that exists in <strong>both</strong> Kafka and Iceberg (e.g.
      <code>truck_positions</code>) is <strong>intelligently unioned by Streambased</strong> —
      live records and archived records returned together in a single query, with no duplicates.
    </div>

  </div>
</div>

"""))

In [ ]:
q("SHOW TABLES IN isk.merged")

### Sample each namespace

For each of `hotset`, `coldset`, and `merged`, we'll pull the newest row (`ORDER BY kafka_offset DESC`) and the oldest row (`ORDER BY kafka_offset ASC`). Comparing those two numbers per namespace is what reveals the boundary between live and archived data — not the row contents themselves.


In [ ]:
q("SELECT * FROM isk.hotset.truck_positions ORDER BY kafka_offset DESC LIMIT 5")

In [ ]:
q("SELECT * FROM isk.hotset.truck_positions ORDER BY kafka_offset ASC LIMIT 5")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <h3 style="font-size: 1.15em; font-weight: 800; color: #1a237e; margin: 0 0 12px 0;">
      The hotset contains a small set of recent offsets
    </h3>

    <p style="font-size: 0.92em; color: #555; line-height: 1.7; margin: 0 0 14px 0;">
      This is because the hotset is served from Kafka and subject to Kafka retention.
    </p>


  </div>
</div>

"""))

In [ ]:
q("SELECT * FROM isk.coldset.truck_positions ORDER BY kafka_offset DESC LIMIT 5")

In [ ]:
q("SELECT * FROM isk.coldset.truck_positions ORDER BY kafka_offset ASC LIMIT 5")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <h3 style="font-size: 1.15em; font-weight: 800; color: #1a237e; margin: 0 0 12px 0;">
      The coldset contains a large set of offsets — from 0 to the start of the hotset
    </h3>

    <p style="font-size: 0.92em; color: #555; line-height: 1.7; margin: 0 0 14px 0;">
      The archived Iceberg data covers the full historical offset range, right from
      offset 0 up to where the hotset begins. 
    </p>

  </div>
</div>

"""))

In [ ]:
q("SELECT * FROM isk.merged.truck_positions ORDER BY kafka_offset DESC LIMIT 5")

In [ ]:
q("SELECT * FROM isk.merged.truck_positions ORDER BY kafka_offset ASC LIMIT 5")

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 20px auto;">
  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px;
              padding: 22px 26px;">

    <h3 style="font-size: 1.15em; font-weight: 800; color: #1a237e; margin: 0 0 12px 0;">
      ⚡ The merged set contains the union of hotset and coldset
    </h3>

    <p style="font-size: 0.92em; color: #555; line-height: 1.7; margin: 0;">
      The queries above show offsets from both ends of the range — low offsets from
      the coldset archive and high offsets from the live hotset. <code>isk.merged</code>
      presents them as a single contiguous table: one query, full history, zero gaps.
    </p>

  </div>
</div>

"""))

<a id="part-4-hot-to-cold-transfer"></a>
## Part 4 — Hot-to-Cold Transfer


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 12px; padding: 28px 32px; text-align: center; color: white;">
  <div style="font-size: 0.72em; letter-spacing: 4px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 10px;">Under the hood</div>
  <h2 style="font-size: 2.2em; font-weight: 900; margin: 0; letter-spacing: -1px;
             line-height: 1.1;">Streambased Hot &#8594; Cold Transfer</h2>
  <div style="width: 48px; height: 3px; background: #4fc3f7; border-radius: 2px;
              margin: 18px auto 0 auto;"></div>
</div>
</div>

"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #fff3e0; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">💰</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Under the hood</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">Why transfer data at all?</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    Kafka storage is fast, but it isn't cheap — keeping every message forever on Kafka brokers
    doesn't scale, so data can't live there indefinitely. At some point it needs to move
    somewhere cheaper for long-term storage.
  </p>

  <p style="margin: 0; line-height: 1.75; color: #37474f;">
    Streambased makes that move simple and efficient: a single SQL statement copies data
    from the hotset straight into Iceberg on object storage, no separate pipeline or ETL
    job required.
  </p>

</div>
</div>
"""))


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<!-- ── FLOW DIAGRAM ─────────────────────────────────────────────────── -->
<div style="display: flex; align-items: center; gap: 0; margin-bottom: 16px;">

  <div style="flex: 1; background: #fff; border: 2px solid #e63946; border-radius: 10px;
              padding: 20px 22px; text-align: center;">
    <div style="font-size: 1.3em; margin-bottom: 6px;">🔴</div>
    <div style="font-family: monospace; font-weight: 700; color: #e63946; font-size: 1em;">isk.hotset</div>
    <div style="font-size: 0.82em; color: #777; margin-top: 6px; line-height: 1.5;">
      Live Kafka events<br>always queryable
    </div>
  </div>

  <div style="flex: 0; padding: 0 12px; text-align: center;">
    <div style="background: #1e1e2e; border-radius: 8px; padding: 10px 14px;
                font-family: monospace; font-size: 0.78em; color: #a6e3a1; white-space: nowrap;">
      INSERT INTO coldset<br>SELECT * FROM hotset
    </div>
    <div style="color: #bbb; font-size: 1.4em; margin-top: 6px;">→</div>
  </div>

  <div style="flex: 1; background: #fff; border: 2px solid #4cc9f0; border-radius: 10px;
              padding: 20px 22px; text-align: center;">
    <div style="font-size: 1.3em; margin-bottom: 6px;">🔵</div>
    <div style="font-family: monospace; font-weight: 700; color: #0096c7; font-size: 1em;">isk.coldset</div>
    <div style="font-size: 0.82em; color: #777; margin-top: 6px; line-height: 1.5;">
      Archived in Iceberg<br>cheap long-term storage
    </div>
  </div>

</div>

<!-- ── TWO CALLOUTS ──────────────────────────────────────────────────── -->
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">

  <div style="background: #e8f5e9; border-left: 5px solid #43a047; border-radius: 8px; padding: 18px 22px;">
    <div style="font-weight: 700; color: #1b5e20; margin-bottom: 8px;">✅ It's just SQL</div>
    <div style="font-size: 0.9em; color: #2e7d32; line-height: 1.7;">
      Moving data from the hotset to the coldset is a single standard SQL statement —
      no custom connectors, no Flink jobs, no bespoke tooling. If you can write
      <code>INSERT INTO … SELECT * FROM …</code> you can archive data to Iceberg.
    </div>
  </div>

  <div style="background: #fff8e1; border-left: 5px solid #f9a825; border-radius: 8px; padding: 18px 22px;">
    <div style="font-weight: 700; color: #e65100; margin-bottom: 8px;">⚡ Not required for availability</div>
    <div style="font-size: 0.9em; color: #bf360c; line-height: 1.7;">
      Crucially, this transfer is <strong>optional and decoupled from data availability</strong>.
      Data in the hotset is already fully queryable via <code>isk.merged</code> — you archive
      to coldset for cost and retention, not because anyone needs it to see the data.
    </div>
  </div>

</div>

</div>

"""))

In [ ]:
show_distribution()

> **Note:** this moves data, it doesn't just read it. Running this cell more than once will insert the same hotset rows into `coldset` again, creating duplicates — if you want to see the transfer again, restart from the top rather than re-running just this cell.


In [ ]:
q("INSERT INTO direct.coldset.truck_positions SELECT * FROM isk.hotset.truck_positions")

In [ ]:
show_distribution()

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #e8f5e9; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">✅</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Under the hood</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">What just happened</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    The <code>INSERT INTO</code> above copied the hotset's <code>truck_positions</code> records
    into <code>coldset</code> — 100% of that data is now durably available from Iceberg, not just
    from Kafka.
  </p>

  <p style="margin: 0; line-height: 1.75; color: #37474f;">
    <strong>Streambased didn't delete anything from Kafka to do this.</strong> The messages are
    still sitting on the topic, exactly as before — operational applications consuming from Kafka
    directly may still need them. Removing them from Kafka is a separate concern, left entirely to
    Kafka's own retention settings, on Kafka's own schedule.
  </p>

</div>
</div>
"""))


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 24px auto;">

  <!-- ── headline ── -->
  <div style="text-align: center; margin-bottom: 20px;">
    <div style="font-size: 0.72em; letter-spacing: 3px; color: #90a4ae;
                text-transform: uppercase; margin-bottom: 6px;">Streambased advantage</div>
    <h3 style="font-size: 1.4em; font-weight: 800; color: #1a237e; margin: 0;">
      Transfer data only when it&#39;s efficient to do so
    </h3>
    <p style="font-size: 0.92em; color: #555; margin: 10px 0 0 0; line-height: 1.6;">
      Because Kafka is always available as a live Iceberg view, you decide
      <em>when</em> to write to object storage — not the pipeline.
    </p>
  </div>

  <!-- ── three boxes ── -->
  <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 14px;">

    <!-- Small files -->
    <div style="background: #fff; border: 2px solid #e63946; border-radius: 12px;
                padding: 22px 20px;">
      <div style="font-size: 1.8em; margin-bottom: 10px;">📂</div>
      <div style="font-weight: 800; color: #c62828; font-size: 1em; margin-bottom: 8px;">
        No more small files
      </div>
      <div style="font-size: 0.86em; color: #555; line-height: 1.65;">
        Traditional streaming writes one Iceberg file per micro-batch — thousands of
        tiny files that bloat your object store and kill scan performance. With
        Streambased you batch writes at your cadence, so every file is right-sized.
      </div>
    </div>

    <!-- Metadata explosion -->
    <div style="background: #fff; border: 2px solid #ff6f00; border-radius: 12px;
                padding: 22px 20px;">
      <div style="font-size: 1.8em; margin-bottom: 10px;">💥</div>
      <div style="font-weight: 800; color: #e65100; font-size: 1em; margin-bottom: 8px;">
        No more metadata explosion
      </div>
      <div style="font-size: 0.86em; color: #555; line-height: 1.65;">
        Every Iceberg commit adds a snapshot, manifest list, and manifest files.
        Continuous streaming turns your catalog into a metadata graveyard.
        Infrequent, batched commits keep the metadata tree small and compaction
        overhead near zero.
      </div>
    </div>

    <!-- Availability -->
    <div style="background: linear-gradient(135deg, #e8f5e9, #e3f2fd);
                border: 2px solid #43a047; border-radius: 12px; padding: 22px 20px;">
      <div style="font-size: 1.8em; margin-bottom: 10px;">✅</div>
      <div style="font-weight: 800; color: #2e7d32; font-size: 1em; margin-bottom: 8px;">
        No Zero availability trade-off
      </div>
      <div style="font-size: 0.86em; color: #555; line-height: 1.65;">
        Data doesn&#39;t disappear while it waits. <code>isk.hotset</code> keeps
        every uncommitted record instantly queryable via the same Iceberg interface —
        so there&#39;s no gap between the last archive and right now.
      </div>
    </div>

  </div>
</div>

"""))

<a id="part-5-kafka-consumers-reading-archived-data"></a>
## Part 5 — Kafka Consumers Reading Archived Data (Slipstream)


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 12px; padding: 28px 32px; text-align: center; color: white;">
  <div style="font-size: 0.72em; letter-spacing: 4px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 10px;">Under the hood</div>
  <h2 style="font-size: 2.2em; font-weight: 900; margin: 0; letter-spacing: -1px;
             line-height: 1.1;">Consuming Iceberg data as Kafka</h2>
  <div style="width: 48px; height: 3px; background: #4fc3f7; border-radius: 2px;
              margin: 18px auto 0 auto;"></div>
</div>
</div>

"""))

### Force Kafka retention to roll off early

In production, Kafka would only delete old messages after the configured retention window (often days). To see that behaviour without waiting, the two cells below temporarily lower the `truck_positions` topic's config directly:

- `segment.ms=500` — closes the active log segment every 500ms instead of waiting for the default schedule, so old segments become eligible for deletion almost immediately.
- `retention.ms=500` — tells Kafka to delete any closed segment older than 500ms.

Together, these force Kafka to delete the topic's oldest messages within seconds. Both settings get restored to their normal values later in this section.

> **Note:** this isn't special Streambased behaviour — it's completely normal Kafka retention. We're just turning the dial up so it happens in seconds instead of days, since a notebook demo can't sit around waiting for a real retention window to pass.


In [ ]:
compose('exec', 'kafka1', 'kafka-configs', '--bootstrap-server', 'kafka1:9092', '--alter', '--topic', 'truck_positions', '--add-config', 'segment.ms=500')

In [ ]:
compose('exec', 'kafka1', 'kafka-configs', '--bootstrap-server', 'kafka1:9092', '--alter', '--topic', 'truck_positions', '--add-config', 'retention.ms=500')

### Confirm retention has kicked in

Rather than checking in an external UI, we can ask `isk.hotset` directly: once Kafka has actually deleted the oldest segments, the earliest visible `kafka_offset` will move past `0`.

This can take up to a minute to show up here even though Kafka has already deleted the data — Streambased caches Kafka offset ranges internally and refreshes that cache roughly every 60 seconds, on top of Spark's own catalog cache. The cell below polls until it sees the change (or times out).


In [ ]:
import time

def wait_for_retention(table='truck_positions', timeout_s=90, poll_s=5):
    start = time.time()
    while time.time() - start < timeout_s:
        q(f"REFRESH TABLE isk.hotset.{table}")
        row = q(f"SELECT MIN(kafka_offset) AS earliest, COUNT(*) AS cnt FROM isk.hotset.{table}").iloc[0]
        earliest, count = row['earliest'], row['cnt']
        elapsed = int(time.time() - start)
        # retention has kicked in once either the earliest visible offset has moved past 0,
        # or — if it deleted everything currently on the topic — the table is just empty
        # (in which case MIN(kafka_offset) is NULL, not > 0, so count==0 must be checked too)
        if count == 0 or (earliest is not None and earliest > 0):
            print(f"Retention has kicked in — earliest visible offset is now {earliest}, {count} row(s) visible (took {elapsed}s)")
            return earliest
        print(f"Still waiting for retention to take effect... earliest offset is {earliest}, {count} row(s) visible ({elapsed}s elapsed)")
        time.sleep(poll_s)
    print("Timed out waiting for retention to take effect — Streambased's offset cache may just need a little longer. Try re-running this cell.")
    return None

wait_for_retention()


### Restore normal settings

Retention has done its job — the cells below just put `retention.ms` and `segment.ms` back to their original 7-day values, so the topic behaves normally again for the rest of the notebook.


In [ ]:
compose('exec', 'kafka1', 'kafka-configs', '--bootstrap-server', 'kafka1:9092', '--alter', '--topic', 'truck_positions', '--add-config', 'retention.ms=604800000')

In [ ]:
compose('exec', 'kafka1', 'kafka-configs', '--bootstrap-server', 'kafka1:9092', '--alter', '--topic', 'truck_positions', '--add-config', 'segment.ms=604800000')

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 24px auto;">

  <!-- ── problem statement ── -->
  <div style="background: #fff8e1; border: 2px solid #f9a825; border-radius: 12px;
              padding: 22px 26px; margin-bottom: 16px; display: flex; gap: 18px; align-items: flex-start;">
    <div style="font-size: 2em; flex-shrink: 0;">⚠️</div>
    <div>
      <div style="font-weight: 800; font-size: 1.05em; color: #e65100; margin-bottom: 6px;">
        A new problem: Kafka clients are now blind
      </div>
      <div style="font-size: 0.9em; color: #555; line-height: 1.7;">
        Moving data from Kafka to Iceberg is efficient for analytics — but it creates
        an availability gap for any downstream system that reads from Kafka directly.
        Once messages age off the topic, consumers running on Kafka offsets
        <strong>can no longer reach that data</strong>. It simply isn&#39;t there.
      </div>
    </div>
  </div>

  <!-- ── two-column before/after ── -->
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 16px;">

    <div style="background: #fff; border: 2px solid #e63946; border-radius: 10px; padding: 20px;">
      <div style="font-weight: 700; color: #c62828; margin-bottom: 10px; font-size: 0.95em;">
        ❌ Without Streambased
      </div>
      <ul style="font-size: 0.87em; color: #555; line-height: 1.75; margin: 0; padding-left: 18px;">
        <li>Kafka consumer reaches end of retention</li>
        <li>Data has been archived to Iceberg</li>
        <li>Consumer gets no more records — offset is stale</li>
        <li>Applications must be rewritten to query Kafka + Iceberg</li>
        <li>Two code paths, two schemas, two maintenance burdens</li>
      </ul>
    </div>

    <div style="background: #e8f5e9; border: 2px solid #43a047; border-radius: 10px; padding: 20px;">
      <div style="font-weight: 700; color: #2e7d32; margin-bottom: 10px; font-size: 0.95em;">
        ✅ With Streambased
      </div>
      <ul style="font-size: 0.87em; color: #555; line-height: 1.75; margin: 0; padding-left: 18px;">
        <li>Iceberg data is <em>re-surfaced</em> as a Kafka topic</li>
        <li>Data is stored cheaply and efficiently</li>
        <li>No code changes, no schema changes</li>
        <li>Kafka clients transparently read through to Iceberg</li>
        <li>One interface — infinite effective retention</li>
      </ul>
    </div>

  </div>

</div>

"""))

In [ ]:
show_distribution()

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #fce4ec; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">🔴</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Under the hood</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">Read the topic with a plain Kafka consumer</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    This runs a real <code>kafka-avro-console-consumer</code> — a standard Kafka CLI client,
    nothing Streambased-specific — pointed straight at the broker (<code>kafka1:9092</code>)
    and asked to read <code>truck_positions</code> <code>--from-beginning</code>.
  </p>

  <p style="margin: 0; line-height: 1.75; color: #37474f;">
    <strong>What to look for:</strong> with <code>print.offset=true</code>, each printed record is
    prefixed with its offset. Before this section, that would start at offset <code>0</code>. Now
    that retention has deleted the earliest segments, "from the beginning" actually starts from
    whatever the new low watermark is — so the first offset printed will be the same non-zero
    number <code>wait_for_retention()</code> reported above, not <code>0</code>. The messages
    before that offset are simply gone from this consumer's point of view.
  </p>

</div>
</div>
"""))


In [ ]:
compose('exec', 'schema-registry', 'kafka-avro-console-consumer', '--topic', 'truck_positions', '--bootstrap-server', 'kafka1:9092', '--from-beginning', '--max-messages', '10', '--property', 'schema.registry.url=http://schema-registry:8081', '--property', 'print.offset=true', '--property', 'print.key=true', '--key-deserializer', 'org.apache.kafka.common.serialization.StringDeserializer')

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 24px auto;">

  <!-- ── headline ── -->
  <div style="text-align: center; margin-bottom: 22px;">
    <div style="font-size: 0.72em; letter-spacing: 3px; color: #90a4ae;
                text-transform: uppercase; margin-bottom: 8px;">Streambased Slipstream</div>
    <h3 style="font-size: 1.5em; font-weight: 800; color: #1a237e; margin: 0 0 10px 0;">
      Iceberg data, surfaced as Kafka
    </h3>
    <p style="font-size: 0.92em; color: #555; max-width: 560px; margin: 0 auto; line-height: 1.65;">
      Streambased proxies the Kafka protocol — intercepting consumer fetch requests
      and transparently serving archived Iceberg records. Your consumer never knows the difference.
    </p>
  </div>

  <!-- ── diagram ── -->
  <div style="background: #f8f9fa; border: 1.5px solid #e0e0e0; border-radius: 14px;
              padding: 28px 20px; margin-bottom: 16px;">
  <svg viewBox="0 0 720 180" xmlns="http://www.w3.org/2000/svg"
       style="width: 100%; max-width: 720px; display: block; margin: 0 auto;">

    <!-- Kafka Consumer -->
    <rect x="10" y="60" width="130" height="60" rx="10" fill="#fff"
          stroke="#43a047" stroke-width="2"/>
    <text x="75" y="84" font-size="11" font-weight="700" fill="#2e7d32"
          font-family="system-ui" text-anchor="middle">Kafka Consumer</text>
    <text x="75" y="100" font-size="10" fill="#888" font-family="monospace"
          text-anchor="middle">unchanged</text>
    <text x="75" y="114" font-size="18" text-anchor="middle">🖥️</text>

    <!-- Arrow: consumer → proxy -->
    <line x1="140" y1="90" x2="198" y2="90" stroke="#90a4ae" stroke-width="1.5"
          marker-end="url(#arr)"/>
    <text x="169" y="84" font-size="9" fill="#90a4ae" font-family="system-ui"
          text-anchor="middle">FETCH req</text>

    <!-- Streambased Proxy box -->
    <rect x="200" y="30" width="160" height="120" rx="12"
          fill="url(#sbg)" stroke="#7b2d8b" stroke-width="2"/>
    <defs>
      <linearGradient id="sbg" x1="0" y1="0" x2="0" y2="1">
        <stop offset="0%" stop-color="#7b2d8b" stop-opacity="0.07"/>
        <stop offset="100%" stop-color="#2c5364" stop-opacity="0.07"/>
      </linearGradient>
      <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
        <path d="M0,0 L0,6 L8,3 z" fill="#90a4ae"/>
      </marker>
      <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
        <path d="M0,0 L0,6 L8,3 z" fill="#4cc9f0"/>
      </marker>
      <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
        <path d="M0,0 L0,6 L8,3 z" fill="#43a047"/>
      </marker>
    </defs>
    <text x="280" y="58" font-size="9" fill="#7b2d8b" font-family="system-ui"
          text-anchor="middle" font-weight="700" letter-spacing="1.5">STREAMBASED</text>
    <text x="280" y="74" font-size="10" font-weight="700" fill="#4a148c"
          font-family="system-ui" text-anchor="middle">Kafka Proxy</text>
    <line x1="210" y1="83" x2="350" y2="83" stroke="#e0e0e0" stroke-width="1"/>
    <text x="280" y="100" font-size="9" fill="#555" font-family="system-ui"
          text-anchor="middle">offset in hotset?</text>
    <text x="280" y="115" font-size="9" fill="#43a047" font-family="system-ui"
          text-anchor="middle" font-weight="600">→ serve from Kafka</text>
    <text x="280" y="130" font-size="9" fill="#0096c7" font-family="system-ui"
          text-anchor="middle" font-weight="600">→ fetch from Iceberg</text>
    <text x="280" y="145" font-size="9" fill="#555" font-family="system-ui"
          text-anchor="middle">return as Kafka records</text>

    <!-- Arrow: proxy → Iceberg -->
    <line x1="360" y1="110" x2="418" y2="130" stroke="#4cc9f0" stroke-width="1.5"
          marker-end="url(#arr-b)" stroke-dasharray="5,3"/>
    <text x="390" y="110" font-size="9" fill="#0096c7" font-family="system-ui"
          text-anchor="middle">Iceberg scan</text>

    <!-- Iceberg box -->
    <rect x="420" y="110" width="130" height="56" rx="10" fill="#fff"
          stroke="#4cc9f0" stroke-width="2"/>
    <text x="485" y="133" font-size="11" font-weight="700" fill="#0096c7"
          font-family="system-ui" text-anchor="middle">Iceberg</text>
    <text x="485" y="149" font-size="10" fill="#888" font-family="system-ui"
          text-anchor="middle">object storage</text>
    <text x="420" y="133" font-size="16" text-anchor="middle">🗄️</text>

    <!-- Arrow: proxy → Kafka (hotset) -->
    <line x1="360" y1="75" x2="418" y2="55" stroke="#43a047" stroke-width="1.5"
          marker-end="url(#arr-g)" stroke-dasharray="5,3"/>
    <text x="393" y="55" font-size="9" fill="#43a047" font-family="system-ui"
          text-anchor="middle">live fetch</text>

    <!-- Kafka (hotset) box -->
    <rect x="420" y="14" width="130" height="56" rx="10" fill="#fff"
          stroke="#e63946" stroke-width="2"/>
    <text x="485" y="37" font-size="11" font-weight="700" fill="#e63946"
          font-family="system-ui" text-anchor="middle">Kafka</text>
    <text x="485" y="53" font-size="10" fill="#888" font-family="system-ui"
          text-anchor="middle">hotset (live)</text>
    <text x="420" y="37" font-size="16" text-anchor="middle">📨</text>

    <!-- Arrow: results back to consumer -->
    <line x1="200" y1="100" x2="142" y2="100" stroke="#7b2d8b" stroke-width="1.5"
          marker-end="url(#arr)"/>
    <text x="169" y="114" font-size="9" fill="#7b2d8b" font-family="system-ui"
          text-anchor="middle">Kafka records</text>

    <!-- Right label -->
    <rect x="565" y="14" width="148" height="152" rx="10" fill="#fffde7"
          stroke="#f9a825" stroke-width="1.5"/>
    <text x="639" y="36" font-size="10" font-weight="700" fill="#e65100"
          font-family="system-ui" text-anchor="middle">Consumer sees</text>
    <line x1="575" y1="44" x2="703" y2="44" stroke="#f9a825" stroke-width="1"/>
    <text x="575" y="62" font-size="9" fill="#555" font-family="system-ui">✅ Same topic name</text>
    <text x="575" y="80" font-size="9" fill="#555" font-family="system-ui">✅ Same Kafka API</text>
    <text x="575" y="98" font-size="9" fill="#555" font-family="system-ui">✅ Continuous offsets</text>
    <text x="575" y="116" font-size="9" fill="#555" font-family="system-ui">✅ Same record schema</text>
    <text x="575" y="140" font-size="9" fill="#888" font-family="system-ui">❌ No code changes</text>
    <text x="575" y="158" font-size="9" fill="#888" font-family="system-ui">❌ No migrations</text>

  </svg>
  </div>

</div>

"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 28px 32px;">

  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 20px;">
    <div style="background: #ede7f6; border-radius: 50%; width: 42px; height: 42px;
                display: flex; align-items: center; justify-content: center; font-size: 1.4em;">⚡</div>
    <div>
      <div style="font-size: 0.78em; color: #90a4ae; letter-spacing: 2px; text-transform: uppercase;">Under the hood</div>
      <div style="font-size: 1.1em; font-weight: 700; color: #1a237e;">Read the same topic through Streambased instead</div>
    </div>
  </div>

  <p style="margin: 0 0 14px 0; line-height: 1.75; color: #37474f;">
    Same command, same topic, same <code>--from-beginning</code> — the only change is the
    bootstrap server: <code>ksi:9192</code> instead of <code>kafka1:9092</code>. That's
    Streambased's KSI component, proxying the Kafka protocol as shown in the diagram above.
  </p>

  <p style="margin: 0; line-height: 1.75; color: #37474f;">
    <strong>What to look for:</strong> the offsets printed this time start back at <code>0</code>.
    Nothing changed on the producer or consumer side — Streambased is transparently serving the
    deleted early messages from Iceberg's <code>coldset</code> and stitching them back in as
    ordinary Kafka records.
  </p>

</div>
</div>
"""))


In [ ]:
compose('exec', 'schema-registry', 'kafka-avro-console-consumer', '--topic', 'truck_positions', '--bootstrap-server', 'ksi:9192', '--from-beginning', '--property', 'schema.registry.url=http://schema-registry:8081', '--property', 'print.offset=true', '--property', 'print.key=true', '--key-deserializer', 'org.apache.kafka.common.serialization.StringDeserializer', '--max-messages', '10')

<a id="part-6-cdc-materialization"></a>
## Part 6 — CDC: Automatic Change-Data-Capture Materialization


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 12px; padding: 28px 32px; text-align: center; color: white;">
  <div style="font-size: 0.72em; letter-spacing: 4px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 10px;">Under the hood</div>
  <h2 style="font-size: 2.2em; font-weight: 900; margin: 0; letter-spacing: -1px;
             line-height: 1.1;">Change data capture, materialized automatically</h2>
  <div style="width: 48px; height: 3px; background: #4fc3f7; border-radius: 2px;
              margin: 18px auto 0 auto;"></div>
</div>
</div>

"""))


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto 24px auto;">

  <!-- ── problem statement ── -->
  <div style="background: #fff8e1; border: 2px solid #f9a825; border-radius: 12px;
              padding: 22px 26px; margin-bottom: 16px; display: flex; gap: 18px; align-items: flex-start;">
    <div style="font-size: 2em; flex-shrink: 0;">🔄</div>
    <div>
      <div style="font-weight: 800; font-size: 1.05em; color: #e65100; margin-bottom: 6px;">
        A change log isn't a table
      </div>
      <div style="font-size: 0.9em; color: #555; line-height: 1.7;">
        The <code>orders</code> topic doesn't carry order records — it carries a stream of
        <strong>change events</strong> in Debezium's envelope format: an <code>op</code> field
        (<code>c</code>/<code>u</code>/<code>d</code>) plus <code>before</code>/<code>after</code>
        payloads. Read naively, the same <code>OrderID</code> shows up over and over as it changes,
        with no single row answering "what does this order look like right now?"
      </div>
    </div>
  </div>

  <!-- ── two-column before/after ── -->
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 16px;">

    <div style="background: #fff; border: 2px solid #e63946; border-radius: 10px; padding: 20px;">
      <div style="font-weight: 700; color: #c62828; margin-bottom: 10px; font-size: 0.95em;">
        ❌ Without Streambased
      </div>
      <ul style="font-size: 0.87em; color: #555; line-height: 1.75; margin: 0; padding-left: 18px;">
        <li>Consumers see raw <code>op</code>/<code>before</code>/<code>after</code> JSON</li>
        <li>Each key appears once per change — duplicated, not deduplicated</li>
        <li>No row represents "current state" on its own</li>
        <li>Need a CDC connector plus a compaction/materialized-view job</li>
        <li>Extra infrastructure just to answer a simple lookup</li>
      </ul>
    </div>

    <div style="background: #e8f5e9; border: 2px solid #43a047; border-radius: 10px; padding: 20px;">
      <div style="font-weight: 700; color: #2e7d32; margin-bottom: 10px; font-size: 0.95em;">
        ✅ With Streambased
      </div>
      <ul style="font-size: 0.87em; color: #555; line-height: 1.75; margin: 0; padding-left: 18px;">
        <li>I.S.K. recognizes the Debezium envelope automatically</li>
        <li>Each <code>op</code> is applied in place, keyed by the record key</li>
        <li><code>isk.hotset.orders</code> always shows one row per key</li>
        <li>No connector, no compaction job, no schema mapping to maintain</li>
        <li>Same physical Kafka topic — just queried as a live table</li>
      </ul>
    </div>

  </div>

</div>

"""))


### Read the raw topic with a plain Kafka consumer

Before looking at the materialized table, let's see what's actually on the wire. This is the
same `kafka-avro-console-consumer` used earlier in the notebook, this time pointed at the
`orders` topic.

**What to look for:** every message carries an `op` field (`c` = create, `u` = update,
`d` = delete) plus `before`/`after` payloads — the raw Debezium envelope. The same
`OrderID` can appear multiple times as its status changes.


In [ ]:
compose('exec', 'schema-registry', 'kafka-avro-console-consumer', '--topic', 'orders', '--bootstrap-server', 'kafka1:9092', '--from-beginning', '--max-messages', '10', '--property', 'schema.registry.url=http://schema-registry:8081', '--property', 'print.offset=true', '--property', 'print.key=true')


### The same topic, unwrapped into a live table

Now query the identical Kafka topic through `isk.hotset` instead. No connector was configured
and no materialized view was built — Streambased recognized the envelope shape and did the
unwrapping automatically.

Notice the schema below: it's the flat `OrderValue` shape (`OrderID`, `CustomerID`, `Status`,
`Amount`), not the raw envelope. And the row count: with only 5 order keys in play, there are
at most 5 rows — one per key, always showing its latest state.

As with the retention cache in Part 5, Streambased refreshes its view of which topics exist on
a short cycle rather than instantaneously — so if `orders` was only just created, the cell below
waits for it to show up rather than failing outright.


In [ ]:
import time

def wait_for_orders_table(timeout_s=90, poll_s=5):
    start = time.time()
    while time.time() - start < timeout_s:
        try:
            spark.sql("REFRESH TABLE isk.hotset.orders")
            count = q("SELECT COUNT(*) AS cnt FROM isk.hotset.orders").iloc[0]['cnt']
            elapsed = int(time.time() - start)
            print(f"isk.hotset.orders is available — {count} row(s) visible (took {elapsed}s)")
            return True
        except Exception:
            elapsed = int(time.time() - start)
            print(f"Still waiting for Streambased to discover the orders topic... ({elapsed}s elapsed)")
            time.sleep(poll_s)
    print("Timed out waiting for isk.hotset.orders to appear — Streambased's topic discovery may just need a little longer. Try re-running this cell.")
    return False

wait_for_orders_table()


In [ ]:
q("DESCRIBE isk.hotset.orders")


In [ ]:
spark.sql("REFRESH TABLE isk.hotset.orders")
q("SELECT OrderID, CustomerID, Status, Amount FROM isk.hotset.orders ORDER BY OrderID")


### Watching rows update in place

The background generator (`live_datagen.py`) cycles through the same 5 order keys, emitting one
create, update, or delete every second. Because I.S.K. applies each change in place, querying
`isk.hotset.orders` repeatedly shows that *same* small set of rows mutating live — no growing
history, no duplicates.

The cell below runs `SELECT * FROM isk.hotset.orders` several times, a couple of seconds apart,
and highlights whatever changed since the previous poll: **yellow** for a row that was created or
updated, **red** for a key that was deleted.


In [ ]:
import time
from datetime import datetime
from IPython.display import display, HTML

def watch_orders_cdc(polls=5, interval_s=2):
    prev = {}
    for i in range(polls):
        spark.sql("REFRESH TABLE isk.hotset.orders")
        df = q("SELECT * FROM isk.hotset.orders ORDER BY OrderID")
        cols = list(df.columns)
        cur = {row['OrderID']: row for row in df.to_dict('records')}

        rows_html = ''
        for oid in sorted(set(prev) | set(cur)):
            if oid in cur:
                row = cur[oid]
                changed = oid not in prev or prev[oid] != row
                bg = 'background:#fff3cd;' if changed else ''
                cells = ''.join(
                    f'<td style="padding:6px 12px;border-bottom:1px solid #eee;">{row[c]}</td>'
                    for c in cols
                )
            else:
                bg = 'background:#ffebee;'
                cells = (
                    f'<td style="padding:6px 12px;border-bottom:1px solid #eee;">{oid}</td>'
                    f'<td colspan="{max(len(cols) - 1, 1)}" style="padding:6px 12px;'
                    f'border-bottom:1px solid #eee;color:#c62828;font-style:italic;">'
                    f'deleted since last poll</td>'
                )
            rows_html += f'<tr style="{bg}">{cells}</tr>'

        if not rows_html:
            rows_html = f'<tr><td colspan="{len(cols)}" style="padding:6px 12px;color:#999;">no rows</td></tr>'

        header_html = ''.join(
            f'<th style="padding:6px 12px;text-align:left;border-bottom:2px solid #ccc;'
            f'font-size:0.82em;color:#555;">{c}</th>'
            for c in cols
        )

        legend = (
            ' &nbsp;·&nbsp; <span style="color:#e65100;font-weight:700;">yellow = changed</span>'
            ' &nbsp;·&nbsp; <span style="color:#c62828;font-weight:700;">red = deleted</span>'
        ) if i > 0 else ''

        html = f'''
        <div style="font-family:system-ui,sans-serif;max-width:860px;margin:0 0 14px 0;">
          <div style="font-size:0.85em;color:#666;margin-bottom:6px;">
            <strong>Poll {i + 1}/{polls}</strong> · {datetime.now().strftime('%H:%M:%S')}{legend}
          </div>
          <table style="border-collapse:collapse;font-size:0.85em;width:100%;">
            <tr>{header_html}</tr>
            {rows_html}
          </table>
        </div>
        '''
        display(HTML(html))
        prev = cur
        if i < polls - 1:
            time.sleep(interval_s)

watch_orders_cdc()


<a id="summary"></a>
## Summary


In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">

<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%);
            border-radius: 16px; padding: 36px 40px; color: white; margin-bottom: 20px; text-align: center;">
  <div style="font-size: 0.78em; letter-spacing: 3px; color: #4fc3f7;
              text-transform: uppercase; margin-bottom: 10px;">What we demonstrated today</div>
  <h2 style="font-size: 2.2em; font-weight: 900; margin: 0 0 12px 0; letter-spacing: -0.5px; line-height: 1.1;">
    The Streambased Advantage
  </h2>
  <p style="margin: 0 auto; font-size: 1em; color: #b0bec5; max-width: 580px; line-height: 1.7;">
    Six capabilities that collapse the gap between streaming data and actionable insight
    with no new pipelines, no ETL jobs, and no data engineering tickets.
  </p>
  <div style="width: 56px; height: 3px; background: #4fc3f7; border-radius: 2px; margin: 20px auto 0 auto;"></div>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 14px;">

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #e3f2fd; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#128268;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">Instant availabiity SQL access to Kafka</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      Every Kafka topic is instantly available as a queryable Iceberg-compatible table via
      <code>isk.hotset</code> &#8212; no connectors, no schema declarations, no pipeline code required.
    </div>
  </div>

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #fce4ec; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#128193;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">Unfied storage &#8212; Kafka + Iceberg</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      A single unified namespace spans three logical tiers: live Kafka data in
      <code>hotset</code>, archived Iceberg data in <code>coldset</code>, and the seamless
      union of both in <code>merged</code>.
    </div>
  </div>

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #e8f5e9; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#9889;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">Real-time meets historical in one query</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      <code>isk.merged</code> lets you benchmark live data against historical baselines in a
      single SQL statement &#8212; no join across systems, no orchestration, no waiting for a batch run.
    </div>
  </div>

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #fff8e1; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#128230;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">Controlled hot&#8594;cold transfer</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      Because Kafka is always available as a live Iceberg view, you choose <em>when</em> to
      commit to object storage. No small files, no metadata explosion &#8212; every write is
      right-sized and on your terms.
    </div>
  </div>

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #ede7f6; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#128737;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">A single source of truth</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      The same unified view presented as <code>Kafka</code> or presented as <code>Iceberg</code>. No duplicated data means no corruption, drift and easy evolution.
    </div>
  </div>

  <div style="background: #fff; border: 1.5px solid #e0e0e0; border-radius: 12px; padding: 24px 26px;">
    <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
      <div style="background: #e0f2f1; border-radius: 50%; width: 40px; height: 40px; flex-shrink: 0;
                  display: flex; align-items: center; justify-content: center; font-size: 1.3em;">&#128260;</div>
      <div style="font-weight: 800; color: #1a237e; font-size: 1em;">Automatic CDC materialization</div>
    </div>
    <div style="font-size: 0.88em; color: #555; line-height: 1.7;">
      Debezium-style change events are recognized and applied in place &#8212; <code>isk.hotset</code>
      always shows current state per key, with no connector or compaction job to run.
    </div>
  </div>

</div>

<div style="background: linear-gradient(135deg, #1b5e20 0%, #2e7d32 60%, #388e3c 100%);
            border-radius: 14px; padding: 28px 36px; color: white;
            display: flex; align-items: center; gap: 28px;">
  <div style="font-size: 4em; line-height: 1; flex-shrink: 0;">&#128640;</div>
  <div>
    <div style="font-size: 0.78em; letter-spacing: 3px; color: #a5d6a7;
                text-transform: uppercase; margin-bottom: 6px;">The bottom line</div>
    <div style="font-size: 1.2em; font-weight: 800; margin-bottom: 8px; line-height: 1.3;">
      One platform. Full data history. No new pipelines.
    </div>
    <div style="font-size: 0.92em; color: #c8e6c9; line-height: 1.7; max-width: 580px;">
      Streambased turns your existing Kafka and Iceberg infrastructure into a unified,
      queryable data platform &#8212; giving every team access to the full picture,
      in real time, from day one.
    </div>
  </div>
</div>

</div>
"""))

In [ ]:
from IPython.display import display, HTML
display(HTML("""
<div style="font-family: system-ui, sans-serif; max-width: 860px; margin: 0 auto;">
<div style="background: linear-gradient(135deg, #1a0533 0%, #3a0ca3 50%, #560bad 100%);
            border-radius: 16px; padding: 36px 40px; color: white; text-align: center;">
  <div style="font-size: 0.78em; letter-spacing: 3px; color: #c77dff;
              text-transform: uppercase; margin-bottom: 10px;">Want this running on your own data?</div>
  <h2 style="font-size: 1.8em; font-weight: 800; margin: 0 0 14px 0; letter-spacing: -0.5px;">
    Talk to Streambased
  </h2>
  <p style="margin: 0 auto 22px auto; font-size: 1em; line-height: 1.7; color: #e0aaff; max-width: 560px;">
    If you'd like to see this working against your own Kafka and Iceberg, or have questions
    about anything in this notebook, we'd love to hear from you.
  </p>
  <a href="https://streambased.io/contact-us" target="_blank"
     style="display: inline-block; background: white; color: #3a0ca3; font-weight: 700;
            padding: 12px 28px; border-radius: 8px; text-decoration: none; font-size: 0.95em;">
    Contact us →
  </a>
</div>
</div>
"""))


<a id="glossary"></a>
## Glossary

Quick reference for terms used throughout this notebook.

| Term | Meaning |
|------|---------|
| `isk` | Streambased's unified catalog — sitting above the pre-existing Kafka and Iceberg environments. It surfaces the three logical views below (`isk.hotset`, `isk.coldset`, `isk.merged`). |
| `direct` | The underlying Iceberg REST catalog itself, This provides direct (and importantly writable) access to the same physical tables as `isk.coldset`. |
| `hotset` (`isk.hotset`) | Live Kafka topics, surfaced as Iceberg-compatible tables with zero lag — as fresh as the last message on the topic. Subject to Kafka's own retention, so it only holds a recent window of offsets. |
| `coldset` (`isk.coldset` / `direct.coldset`) | Historical data archived into Iceberg tables on object storage. Cheap and durable, and covers the full offset history from 0 up to wherever the hotset begins. |
| `merged` (`isk.merged`) | The unified view — hotset and coldset stitched into a single table with no duplicates, so one query spans live and archived data together. |
| `kafka_partition` | A column giving each record's original Kafka partition. |
| `kafka_offset` | A column on these tables giving each record's position in its original Kafka partition. Used throughout the notebook to show where the coldset ends and the hotset begins. |
| `op` / `before` / `after` | The three fields of a Debezium-style CDC envelope on the `orders` topic: `op` is `c` (create), `u` (update), or `d` (delete); `before`/`after` hold the row's prior/new state. `isk.hotset.orders` applies these in place, so it always shows one current-state row per key rather than the raw event stream. |
